# 🚲 Citi Bike Pipeline Analysis

This notebook lets you:
1. **Inspect raw data** ingested into `raw.station_status` and `raw.station_information`
2. **Trace data through each dbt layer** (`raw` → `intermediate` → `consumption`)
3. **Diagnose issues** with `consumption.con_station_status_current` and `consumption.con_station_availability_history`
4. **Compare against the live API** to spot discrepancies

---

## Setup

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import requests

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 120)

load_dotenv('secrets.env')
load_dotenv('settings.env', override=True)

# Tracked station IDs from app/config/groups.yaml
TRACKED_STATIONS = {
    '66de482a-0aca-11e7-82f6-3863bb44ef7c': 'Home #1',
    '66de4897-0aca-11e7-82f6-3863bb44ef7c': 'Home #2',
    '66de1295-0aca-11e7-82f6-3863bb44ef7c': 'Home #3',
    '66de4078-0aca-11e7-82f6-3863bb44ef7c': 'Subway / Gym',
    '90c35466-db3c-4b0d-993e-6e92883773b4': 'Work #1',
    '66db2afe-0aca-11e7-82f6-3863bb44ef7c': 'Work #2',
}

print('Libraries loaded ✓')

In [ ]:
# -- Database Connection ------------------------------------------------------
DB_HOST     = os.getenv('DB_HOST')
DB_PORT     = os.getenv('DB_PORT', '5432')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_NAME     = os.getenv('DB_NAME')

conn_str = f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(conn_str, connect_args={'connect_timeout': 10})

# Quick connection test
with engine.connect() as c:
    result = c.execute(text('SELECT version()')).fetchone()
    print(f'Connected ✓  →  {result[0][:60]}...')

def query(sql, params=None):
    """Helper: run SQL and return a DataFrame.

    Works with SQLAlchemy 1.x and 2.x regardless of pandas version,
    by executing through the engine directly and building the DataFrame manually.
    """
    with engine.connect() as conn:
        result = conn.execute(text(sql), params or {})
        return pd.DataFrame(result.fetchall(), columns=result.keys())

---
## 1 · Raw Layer — `raw.station_status`

In [ ]:
# -- Volume & freshness -------------------------------------------------------
raw_meta = query("""
    SELECT
        COUNT(*)                                        AS total_rows,
        COUNT(DISTINCT station_id)                      AS unique_stations,
        MIN(ingested_at)                                AS earliest_ingest,
        MAX(ingested_at)                                AS latest_ingest,
        NOW() - MAX(ingested_at)                        AS data_age,
        MIN(last_reported)                              AS earliest_reported,
        MAX(last_reported)                              AS latest_reported,
        COUNT(*) FILTER (WHERE last_reported IS NULL)   AS null_last_reported
    FROM raw.station_status
""")
raw_meta.T

In [ ]:
# -- Sample raw rows ----------------------------------------------------------
raw_sample = query("""
    SELECT
        station_id, num_bikes_available, num_docks_available,
        num_bikes_disabled, num_docks_disabled,
        is_installed, is_renting, is_returning,
        last_reported, ingested_at
    FROM raw.station_status
    ORDER BY ingested_at DESC
    LIMIT 20
""")
raw_sample

In [ ]:
# -- Duplicate detection: same (station_id, last_reported) ingested multiple times --
duplicates = query("""
    SELECT
        station_id,
        last_reported,
        COUNT(*) AS num_copies
    FROM raw.station_status
    GROUP BY station_id, last_reported
    HAVING COUNT(*) > 1
    ORDER BY num_copies DESC
    LIMIT 30
""")
print(f'Duplicate (station_id, last_reported) pairs: {len(duplicates)}')
duplicates.head(10)

In [ ]:
# -- Ingestion cadence: how many rows per ingest run? -------------------------
ingest_runs = query("""
    SELECT
        DATE_TRUNC('minute', ingested_at) AS ingest_minute,
        COUNT(*)                          AS rows_inserted
    FROM raw.station_status
    GROUP BY 1
    ORDER BY 1 DESC
    LIMIT 48
""")
print(f'Most recent {len(ingest_runs)} ingest batches:')
ingest_runs.head(10)

In [ ]:
# -- Plot: rows per ingest run over time --------------------------------------
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(ingest_runs['ingest_minute'], ingest_runs['rows_inserted'], width=0.003, color='steelblue')
ax.set_title('Rows inserted per ingest run (recent 48 batches)', fontsize=13)
ax.set_xlabel('Ingest time')
ax.set_ylabel('Row count')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# -- Tracked stations: how many times each has been ingested? -----------------
station_ids_sql = ', '.join(f"'{s}'" for s in TRACKED_STATIONS)
tracked_raw = query(f"""
    SELECT
        station_id,
        COUNT(*)            AS total_rows,
        MIN(last_reported)  AS first_seen,
        MAX(last_reported)  AS last_seen,
        MAX(ingested_at)    AS last_ingested
    FROM raw.station_status
    WHERE station_id IN ({station_ids_sql})
    GROUP BY station_id
    ORDER BY station_id
""")
tracked_raw['alias'] = tracked_raw['station_id'].map(TRACKED_STATIONS)
tracked_raw[['alias', 'station_id', 'total_rows', 'first_seen', 'last_seen', 'last_ingested']]

---
## 2 · Raw Layer — `raw.station_information`

In [ ]:
raw_info = query("""
    SELECT station_id, name, lat, lon, capacity, ingested_at
    FROM raw.station_information
    ORDER BY ingested_at DESC
    LIMIT 20
""")
print(f'Rows in raw.station_information: {len(raw_info)}')
raw_info

In [ ]:
# -- Are tracked stations present in raw.station_information? -----------------
info_tracked = query(f"""
    SELECT station_id, name, capacity
    FROM raw.station_information
    WHERE station_id IN ({station_ids_sql})
""")
info_tracked['alias'] = info_tracked['station_id'].map(TRACKED_STATIONS)

# Show any missing stations
found_ids = set(info_tracked['station_id'])
missing_ids = set(TRACKED_STATIONS.keys()) - found_ids
print(f'Found {len(found_ids)}/{len(TRACKED_STATIONS)} tracked stations in station_information')
if missing_ids:
    print(f'  ⚠️  MISSING from station_information: {missing_ids}')
info_tracked[['alias', 'station_id', 'name', 'capacity']]

---
## 3 · dbt Raw Layer — replicated views

These are straight pass-through views (`raw.raw_station_status`, `raw.raw_station_information`).

In [ ]:
# -- Confirm dbt raw views exist and match the source tables ------------------
dbt_raw_count = query("""
    SELECT
        'raw.raw_station_status (dbt view)'    AS layer,
        COUNT(*)                               AS row_count
    FROM raw.raw_station_status
    UNION ALL
    SELECT
        'raw.station_status (source table)',
        COUNT(*)
    FROM raw.station_status
    UNION ALL
    SELECT
        'raw.raw_station_information (dbt view)',
        COUNT(*)
    FROM raw.raw_station_information
    UNION ALL
    SELECT
        'raw.station_information (source table)',
        COUNT(*)
    FROM raw.station_information
""")
dbt_raw_count

---
## 4 · dbt Intermediate Layer — `intermediate.int_station_status`

This view deduplicates `(station_id, last_reported)` pairs via `DISTINCT ON` and joins with station info.

In [ ]:
# -- Volume after deduplication -----------------------------------------------
int_meta = query("""
    SELECT
        COUNT(*)                    AS total_rows,
        COUNT(DISTINCT station_id)  AS unique_stations,
        MIN(last_reported)          AS earliest,
        MAX(last_reported)          AS latest,
        COUNT(*) FILTER (WHERE station_name IS NULL) AS null_station_names
    FROM intermediate.int_station_status
""")
int_meta.T

In [ ]:
# -- Compare: raw rows vs deduplicated rows per tracked station ---------------
dedup_comparison = query(f"""
    SELECT
        r.station_id,
        COUNT(*) FILTER (WHERE src = 'raw')  AS raw_rows,
        COUNT(*) FILTER (WHERE src = 'int')  AS int_rows
    FROM (
        SELECT station_id, 'raw' AS src FROM raw.station_status
        WHERE station_id IN ({station_ids_sql})
        UNION ALL
        SELECT station_id, 'int' AS src FROM intermediate.int_station_status
        WHERE station_id IN ({station_ids_sql})
    ) r
    GROUP BY r.station_id
    ORDER BY r.station_id
""")
dedup_comparison['alias'] = dedup_comparison['station_id'].map(TRACKED_STATIONS)
dedup_comparison['duplication_factor'] = (dedup_comparison['raw_rows'] / dedup_comparison['int_rows'].replace(0, np.nan)).round(2)
dedup_comparison[['alias', 'raw_rows', 'int_rows', 'duplication_factor']]

In [ ]:
# -- Check: any JOIN misses? (status rows for tracked stations with no name) --
join_misses = query(f"""
    SELECT station_id, COUNT(*) AS rows_without_name
    FROM intermediate.int_station_status
    WHERE station_name IS NULL
      AND station_id IN ({station_ids_sql})
    GROUP BY station_id
""")
if join_misses.empty:
    print('✅ All tracked stations have station_name in the intermediate layer')
else:
    print('⚠️  Tracked stations missing station_name:')
    join_misses['alias'] = join_misses['station_id'].map(TRACKED_STATIONS)
    print(join_misses)

---
## 5 · dbt Consumption Layer — Current Status Table

`consumption.con_station_status_current` — should have **one row per station** with the most recent `last_reported` value.

In [ ]:
# -- Full consumption current snapshot ----------------------------------------
con_current = query("""
    SELECT *
    FROM consumption.con_station_status_current
    ORDER BY last_reported DESC
""")
print(f'Rows in con_station_status_current: {len(con_current)} (should equal unique station count)')
con_current.head(10)

In [ ]:
# -- Diagnose: are tracked stations present? ----------------------------------
con_tracked = con_current[con_current['station_id'].isin(TRACKED_STATIONS)].copy()
con_tracked['alias'] = con_tracked['station_id'].map(TRACKED_STATIONS)

missing_from_con = set(TRACKED_STATIONS.keys()) - set(con_tracked['station_id'])
print(f'Tracked stations in consumption current: {len(con_tracked)}/{len(TRACKED_STATIONS)}')
if missing_from_con:
    print(f'  ⛔  MISSING: {missing_from_con}')

con_tracked[['alias', 'station_name', 'num_bikes_available', 'num_docks_available', 'last_reported']]

In [ ]:
# -- Staleness check: how old is the "current" data? --------------------------
staleness = query("""
    SELECT
        station_id,
        station_name,
        last_reported,
        NOW() - last_reported AS age,
        EXTRACT(EPOCH FROM (NOW() - last_reported))/60 AS age_minutes
    FROM consumption.con_station_status_current
    ORDER BY age DESC
    LIMIT 20
""")
staleness['alias'] = staleness['station_id'].map(lambda x: TRACKED_STATIONS.get(x, ''))
print(f'Most stale stations:')
staleness[['alias', 'station_name', 'last_reported', 'age_minutes']].round(1)

In [ ]:
# -- Key diagnostic: compare what the webapp query returns vs what's in the table --
# This replicates the exact query in app/main.py get_current_availability()
webapp_query_result = query("""
    SELECT * FROM consumption.con_station_status_current
""")

webapp_tracked = webapp_query_result[webapp_query_result['station_id'].isin(TRACKED_STATIONS)].copy()
webapp_tracked['alias'] = webapp_tracked['station_id'].map(TRACKED_STATIONS)

print(f'--- Webapp /api/availability/current would return ---')
print(f'Total rows from DB: {len(webapp_query_result)}')
print(f'Tracked station rows: {len(webapp_tracked)}')
webapp_tracked[['alias', 'station_id', 'station_name', 'num_bikes_available', 'num_docks_available', 'last_reported']]

---
## 6 · dbt Consumption Layer — History Table

`consumption.con_station_availability_history` filters for the past 8 days.

In [ ]:
# -- Volume & date range ------------------------------------------------------
hist_meta = query("""
    SELECT
        COUNT(*)                    AS total_rows,
        COUNT(DISTINCT station_id)  AS unique_stations,
        MIN(last_reported)          AS earliest,
        MAX(last_reported)          AS latest,
        CURRENT_DATE - INTERVAL '8 days' AS window_start
    FROM consumption.con_station_availability_history
""")
hist_meta.T

In [ ]:
# -- Per-tracked-station history row count ------------------------------------
hist_counts = query(f"""
    SELECT
        station_id,
        station_name,
        COUNT(*)            AS history_rows,
        MIN(last_reported)  AS first,
        MAX(last_reported)  AS last
    FROM consumption.con_station_availability_history
    WHERE station_id IN ({station_ids_sql})
    GROUP BY station_id, station_name
    ORDER BY station_id
""")
hist_counts['alias'] = hist_counts['station_id'].map(TRACKED_STATIONS)
hist_counts[['alias', 'station_name', 'history_rows', 'first', 'last']]

In [ ]:
# -- Plot: availability history for tracked stations ---------------------------
hist_data = query(f"""
    SELECT station_id, station_name, last_reported,
           num_bikes_available, num_docks_available
    FROM consumption.con_station_availability_history
    WHERE station_id IN ({station_ids_sql})
    ORDER BY station_id, last_reported
""")
hist_data['alias'] = hist_data['station_id'].map(TRACKED_STATIONS)

aliases = hist_data['alias'].unique()
n = len(aliases)
fig, axes = plt.subplots(n, 1, figsize=(14, 3.5 * n), sharex=True)
if n == 1:
    axes = [axes]

for ax, alias in zip(axes, aliases):
    d = hist_data[hist_data['alias'] == alias]
    ax.plot(d['last_reported'], d['num_bikes_available'], label='Bikes available', color='steelblue')
    ax.plot(d['last_reported'], d['num_docks_available'], label='Docks available', color='darkorange', alpha=0.7)
    ax.set_title(f'{alias}  ({d["station_name"].iloc[0] if len(d) > 0 else "?"})', fontsize=11)
    ax.set_ylabel('Count')
    ax.legend(loc='upper right', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
    ax.grid(alpha=0.3)

plt.xlabel('last_reported')
plt.xticks(rotation=45)
plt.suptitle('Bike / Dock Availability History (consumption layer, 8-day window)', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

---
## 7 · End-to-End Pipeline Health Check

A single diagnostic table comparing row counts at every layer.

In [ ]:
# -- Row count waterfall across all layers ------------------------------------
waterfall = query("""
    SELECT '1. raw.station_status (source)'               AS layer, COUNT(*) AS rows FROM raw.station_status
    UNION ALL
    SELECT '2. raw.raw_station_status (dbt view)',         COUNT(*) FROM raw.raw_station_status
    UNION ALL
    SELECT '3. intermediate.int_station_status (deduped)', COUNT(*) FROM intermediate.int_station_status
    UNION ALL
    SELECT '4. consumption.con_station_status_current',    COUNT(*) FROM consumption.con_station_status_current
    UNION ALL
    SELECT '5. consumption.con_station_availability_history (8d)', COUNT(*) FROM consumption.con_station_availability_history
    ORDER BY layer
""")
waterfall

In [ ]:
# -- Check: does the consumption TABLE need a dbt refresh? --------------------
# The consumption models are materialized as TABLES – they won't update until dbt runs!
# Compare max(last_reported) in intermediate vs consumption current.
freshness_gap = query("""
    SELECT
        MAX(i.last_reported) AS int_max_last_reported,
        MAX(c.last_reported) AS con_max_last_reported,
        MAX(i.last_reported) - MAX(c.last_reported) AS gap
    FROM intermediate.int_station_status i,
         consumption.con_station_status_current c
""")
gap = freshness_gap['gap'].iloc[0]
print(freshness_gap.T)
print()
if gap and gap.total_seconds() > 300:
    print(f'⚠️  STALE TABLE DETECTED: consumption layer is {gap} behind the intermediate layer.')
    print('   Run `dbt run` to refresh the consumption tables.')
else:
    print('✅ Consumption table freshness looks OK')

In [ ]:
# -- Check: ROW_NUMBER() behavior – are there ties on last_reported? ----------
# con_station_status_current uses ROW_NUMBER() OVER (PARTITION BY station_id ORDER BY last_reported DESC)
# If two rows share the exact same last_reported, the result is non-deterministic.
ties = query("""
    SELECT
        station_id,
        last_reported,
        COUNT(*) AS tied_rows
    FROM intermediate.int_station_status
    WHERE (station_id, last_reported) IN (
        SELECT station_id, MAX(last_reported)
        FROM intermediate.int_station_status
        GROUP BY station_id
    )
    GROUP BY station_id, last_reported
    HAVING COUNT(*) > 1
    ORDER BY tied_rows DESC
""")
if ties.empty:
    print('✅ No ties on max(last_reported) per station in intermediate layer')
else:
    print(f'⚠️  {len(ties)} stations have TIED last_reported — ROW_NUMBER() is non-deterministic for these!')
    ties['alias'] = ties['station_id'].map(lambda x: TRACKED_STATIONS.get(x, ''))
    print(ties.head(10))

---
## 8 · Live API Comparison

Fetch directly from the Citi Bike GBFS API and compare against the database.

In [ ]:
# -- Fetch live station status -------------------------------------------------
resp = requests.get('https://gbfs.citibikenyc.com/gbfs/en/station_status.json', timeout=10)
resp.raise_for_status()
live_stations = resp.json()['data']['stations']
live_df = pd.DataFrame(live_stations)
live_df['last_reported'] = pd.to_datetime(live_df['last_reported'], unit='s')
print(f'Live API: {len(live_df)} stations, feed last updated: {live_df["last_reported"].max()}')

live_tracked = live_df[live_df['station_id'].isin(TRACKED_STATIONS)].copy()
live_tracked['alias'] = live_tracked['station_id'].map(TRACKED_STATIONS)
live_tracked[['alias', 'station_id', 'num_bikes_available', 'num_docks_available', 'last_reported']]

In [ ]:
# -- Side-by-side: Live API vs consumption table ------------------------------
comparison = live_tracked[['station_id', 'alias', 'num_bikes_available', 'num_docks_available', 'last_reported']].copy()
comparison.columns = ['station_id', 'alias', 'live_bikes', 'live_docks', 'live_last_reported']

con_merge = con_current[con_current['station_id'].isin(TRACKED_STATIONS)][['station_id', 'num_bikes_available', 'num_docks_available', 'last_reported']].copy()
con_merge.columns = ['station_id', 'con_bikes', 'con_docks', 'con_last_reported']

side_by_side = comparison.merge(con_merge, on='station_id', how='outer')
side_by_side['bikes_diff'] = side_by_side['live_bikes'] - side_by_side['con_bikes']
side_by_side['staleness_min'] = ((side_by_side['live_last_reported'] - side_by_side['con_last_reported'])
                                  .dt.total_seconds() / 60).round(1)

print('Live API vs Consumption Table (for tracked stations):')
side_by_side[['alias', 'live_bikes', 'con_bikes', 'bikes_diff', 'staleness_min', 'live_last_reported', 'con_last_reported']]

---
## 9 · Raw JSON Inspection

Peek inside `raw_json` to see what the API actually returned — useful for catching upstream changes.

In [ ]:
# -- Parse one raw_json blob per tracked station -------------------------------
raw_json_sample = query(f"""
    SELECT DISTINCT ON (station_id)
        station_id,
        raw_json,
        ingested_at
    FROM raw.station_status
    WHERE station_id IN ({station_ids_sql})
    ORDER BY station_id, ingested_at DESC
""")

for _, row in raw_json_sample.iterrows():
    alias = TRACKED_STATIONS.get(row['station_id'], row['station_id'])
    blob = row['raw_json'] if isinstance(row['raw_json'], dict) else json.loads(row['raw_json'])
    print(f'\n-- {alias} (ingested {row["ingested_at"]}) --')
    print(json.dumps(blob, indent=2))

In [ ]:
# -- Detect unexpected keys in raw_json that aren't in the schema -------------
schema_keys = {'station_id', 'num_bikes_available', 'num_bikes_disabled',
               'num_docks_available', 'num_docks_disabled', 'is_installed',
               'is_renting', 'is_returning', 'last_reported'}

sample_row = query("""
    SELECT raw_json FROM raw.station_status ORDER BY ingested_at DESC LIMIT 1
""")
sample_blob = sample_row['raw_json'].iloc[0]
if isinstance(sample_blob, str):
    sample_blob = json.loads(sample_blob)

api_keys = set(sample_blob.keys())
extra_keys = api_keys - schema_keys
missing_keys = schema_keys - api_keys

print('Keys in raw_json but NOT in the table schema (possibly useful but untracked):')
print(f'  Extra:   {extra_keys}')
print(f'  Missing from API (in schema but absent from API): {missing_keys}')

---
## 10 · Summary Dashboard

In [ ]:
print('=' * 60)
print('  PIPELINE HEALTH SUMMARY')
print('=' * 60)

meta = query("""
    SELECT
        (SELECT COUNT(*) FROM raw.station_status)                         AS raw_status_rows,
        (SELECT COUNT(*) FROM raw.station_information)                    AS raw_info_rows,
        (SELECT COUNT(*) FROM intermediate.int_station_status)            AS int_rows,
        (SELECT COUNT(*) FROM consumption.con_station_status_current)     AS con_current_rows,
        (SELECT COUNT(*) FROM consumption.con_station_availability_history) AS con_history_rows,
        (SELECT MAX(ingested_at) FROM raw.station_status)                 AS last_ingest,
        (SELECT MAX(last_reported) FROM consumption.con_station_status_current) AS con_freshness
""")

r = meta.iloc[0]
print(f"  Raw station_status rows   : {r['raw_status_rows']:,}")
print(f"  Raw station_info rows     : {r['raw_info_rows']:,}")
print(f"  Intermediate rows         : {r['int_rows']:,}")
print(f"  Consumption current rows  : {r['con_current_rows']:,}")
print(f"  Consumption history rows  : {r['con_history_rows']:,}")
print(f"  Last ingest ran           : {r['last_ingest']}")
print(f"  Consumption table freshness: {r['con_freshness']}")
print()

n_tracked_in_con = len(con_current[con_current['station_id'].isin(TRACKED_STATIONS)])
n_tracked_total  = len(TRACKED_STATIONS)
status = '✅' if n_tracked_in_con == n_tracked_total else '⛔'
print(f"  {status} Tracked stations in consumption: {n_tracked_in_con}/{n_tracked_total}")

gap_val = freshness_gap['gap'].iloc[0]
if gap_val and gap_val.total_seconds() > 300:
    print(f"  ⚠️  Consumption tables are STALE by {gap_val} — run `dbt run`")
else:
    print(f"  ✅ Consumption tables are fresh")
print('=' * 60)